<a href="https://colab.research.google.com/github/Atherizz/PCVK_2026/blob/main/week_9/tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
# TUGAS PRAKTIKUM MODUL 9 - PENINGKATAN KETERBACAAN TEKS KTP DENGAN OPERASI MORFOLOGI
**Mata Kuliah:** Pengolahan Citra dan Visi Komputer (PCVK)  
**Jurusan:** Teknologi Informasi - D4 Teknik Informatika  
**Topik Tugas:** Implementasi Morfologi Citra (Black-Hat, Opening, Closing, Dilation) pada Dataset KTP untuk Optimalisasi Keterbacaan Teks

## Cell Identitas Mahasiswa

In [ ]:
NAMA = "Savero Athallah Hardiana Putra"
NIM  = "244107020116"
KELAS = "TI-3B"
MATA_KULIAH = "Pengolahan Citra dan Visi Komputer"

print("=" * 55)
print(f"Nama        : {NAMA}")
print(f"NIM         : {NIM}")
print(f"Kelas       : {KELAS}")
print(f"Mata Kuliah : {MATA_KULIAH}")
print("=" * 55)

## Persiapan: Setup Library & Environment Adaptif
Dapat dieksekusi langsung di Google Colab maupun secara lokal.

In [ ]:
import os
import sys
import cv2
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Cek ketersediaan Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("Berjalan di Google Colab environment")
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    print("Berjalan di Local environment (.venv)")

# Penentuan path citra adaptif
def get_img_path(filename):
    candidates = [
        os.path.join("week_9/images", filename),
        os.path.join("images", filename),
        os.path.join("/content/drive/MyDrive/PCVK/Images", filename),
        os.path.join("/content/PCVK_2026/week_9/images", filename),
        os.path.join("week_7/images", filename),
        filename
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return f"/content/drive/MyDrive/PCVK/Images/{filename}"

print("Status environment siap.")

## 1. Analisis Masalah & Eksplorasi Awal Citra KTP
Citra KTP (`ktp_fulan.jpg`) memiliki karakteristik kompleks:
- Teks informasi (NIK, Nama, Alamat, dll.) memiliki intensitas gelap.
- Latar belakang memiliki gradien warna biru dan corak watermark burung garuda.
- Pada binarisasi murni (Modul 7 Otsu), watermark dan variasi pencahayaan ikut terpotong sehingga menghasilkan banyak bintik noise hitam/putih dan goresan huruf yang patah (*broken characters*).

In [ ]:
# Load citra KTP asli
img_ktp = cv2.imread(get_img_path('ktp_fulan.jpg'))
ktp_gray = cv2.cvtColor(img_ktp, cv2.COLOR_BGR2GRAY)

# Baseline Thresholding Otsu murni (Metode Modul 7)
_, ktp_otsu_base = cv2.threshold(ktp_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

plt.figure(figsize=(14, 5))
plt.subplot(121), plt.imshow(cv2.cvtColor(img_ktp, cv2.COLOR_BGR2RGB)), plt.title('Citra KTP Asli'), plt.axis('off')
plt.subplot(122), plt.imshow(ktp_otsu_base, cmap='gray'), plt.title('Baseline Modul 7: Otsu Tanpa Morfologi (Banyak Noise & Bocor)'), plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Implementasi Pipeline Morfologi Komprehensif
Untuk mengatasi masalah di atas, dirancang pipeline morfologi 5 tahap:
1. **Black-Hat Transform ($19 \times 19$):** Mengisolasi teks gelap dan meratakan gradien iluminasi latar belakang.
2. **Binarisasi Otsu:** Mengubah citra hasil Black-Hat menjadi biner secara otomatis.
3. **Morfologi Opening ($2 \times 2$):** Mengikis dan melenyapkan bintik-bintik noise putih di latar belakang.
4. **Morfologi Closing ($2 \times 2$):** Menyambungkan kembali goresan karakter yang patah (*stroke reconstruction*).
5. **Dilation / Thickening ($2 \times 1$):** Penguatan kontur vertikal huruf agar karakter solid, jelas, dan kontras tinggi.

In [ ]:
# 1. Tahap Black-Hat Transform
kernel_bg = cv2.getStructuringElement(cv2.MORPH_RECT, (19, 19))
ktp_blackhat = cv2.morphologyEx(ktp_gray, cv2.MORPH_BLACKHAT, kernel_bg)

# 2. Tahap Binarisasi Otsu
_, ktp_bin = cv2.threshold(ktp_blackhat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# 3. Tahap Morfologi Opening (Pembersihan Noise Latar)
se_open = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
ktp_open = cv2.morphologyEx(ktp_bin, cv2.MORPH_OPEN, se_open)

# 4. Tahap Morfologi Closing (Penyambungan Huruf Terputus)
se_close = cv2.getStructuringElement(cv2.MORPH_RECT, (2, 2))
ktp_close = cv2.morphologyEx(ktp_open, cv2.MORPH_CLOSE, se_close)

# 5. Tahap Dilasi / Thickening (Penguatan Huruf)
se_thick = np.ones((2, 1), dtype=np.uint8)
ktp_final_white_fg = cv2.dilate(ktp_close, se_thick, iterations=1)

# Format Dokumen Resmi (Inversi: Teks Hitam di Latar Putih)
ktp_final_doc = cv2.bitwise_not(ktp_final_white_fg)

# Visualisasi Step-by-Step Pipeline Morfologi
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
axes[0, 0].imshow(cv2.cvtColor(img_ktp, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title('(a) Citra Asli KTP')
axes[0, 0].axis('off')

axes[0, 1].imshow(ktp_blackhat, cmap='gray')
axes[0, 1].set_title('(b) Langkah 1: Black-Hat Transform (Iluminasi Latar Lenyap)')
axes[0, 1].axis('off')

axes[1, 0].imshow(ktp_bin, cmap='gray')
axes[1, 0].set_title('(c) Langkah 2: Binarisasi Otsu')
axes[1, 0].axis('off')

axes[1, 1].imshow(ktp_open, cmap='gray')
axes[1, 1].set_title('(d) Langkah 3: Opening (Bintik Noise Latar Bersih)')
axes[1, 1].axis('off')

axes[2, 0].imshow(ktp_final_white_fg, cmap='gray')
axes[2, 0].set_title('(e) Langkah 4 & 5: Closing + Dilasi (Karakter Solid & Terhubung)')
axes[2, 0].axis('off')

axes[2, 1].imshow(ktp_final_doc, cmap='gray')
axes[2, 1].set_title('(f) Hasil Akhir: Format Dokumen Standar (Sangat Jelas & Tajam)')
axes[2, 1].axis('off')

plt.tight_layout()
plt.show()

## 3. Inspeksi Detail (Zoom-in Region NIK & Nama)
Membandingkan kualitas ketajaman teks NIK dan Nama antara metode Baseline Thresholding (Modul 7) dengan Morfologi Pipeline (Modul 9).

In [ ]:
# Area ROI Teks NIK & Nama (y: 100-200, x: 180-600)
roi_y1, roi_y2 = 100, 200
roi_x1, roi_x2 = 180, 600

crop_orig = img_ktp[roi_y1:roi_y2, roi_x1:roi_x2]
crop_otsu = ktp_otsu_base[roi_y1:roi_y2, roi_x1:roi_x2]
crop_morph = ktp_final_doc[roi_y1:roi_y2, roi_x1:roi_x2]

plt.figure(figsize=(15, 6))
plt.subplot(131), plt.imshow(cv2.cvtColor(crop_orig, cv2.COLOR_BGR2RGB)), plt.title('ROI Citra Asli'), plt.axis('off')
plt.subplot(132), plt.imshow(crop_otsu, cmap='gray'), plt.title('ROI Otsu Modul 7 (Noise & Patah)'), plt.axis('off')
plt.subplot(133), plt.imshow(crop_morph, cmap='gray'), plt.title('ROI Morfologi Modul 9 (Solid & Bersih)'), plt.axis('off')
plt.tight_layout()
plt.show()

## 4. Pembahasan & Jawaban Pertanyaan Analisis (Singkat & To-The-Point)

### 1. Mengapa Operasi Black-Hat Transform Sangat Kritis pada Citra KTP?
- **Jawaban:** Karakter teks pada KTP adalah objek gelap di atas latar belakang yang lebih terang ($f_{teks} < f_{bg}$). Operasi Black-Hat ($(f \bullet b) - f$) mengekstrak elemen yang lebih gelap daripada tetangga lokalnya dalam radius Structuring Element ($19 \times 19$). Efeknya, seluruh variasi gradien warna dan watermark skala besar tereliminasi, menghasilkan kontras teks yang merata di seluruh permukaan KTP.

### 2. Mengapa Dibutuhkan Kombinasi Opening Dilanjutkan Closing?
- **Opening ($2 \times 2$):** Erosi diikuti Dilasi. Mengikis dan menghilangkan bintik-bintik putih (*salt noise*) yang tersisa pada latar belakang tanpa merusak struktur utama huruf.
- **Closing ($2 \times 2$):** Dilasi diikuti Erosi. Menutup celah kecil dan menyambungkan kembali garis huruf yang terputus (*broken strokes*) akibat kontras rendah.

### 3. Apa Pengaruh Structuring Element Persegi Panjang Vertikal ($2 \times 1$) pada Tahap Akhir?
- **Jawaban:** Structuring element berorientasi vertikal ($2 \times 1$) mempertebal goresan vertikal huruf tanpa membuat huruf-huruf yang berdekatan horizontal (spasi antar huruf) saling bertabrakan atau menyatu (*character merging*).

### 4. Kesimpulan Perbandingan Modul 7 vs Modul 9:
| Parameter | Thresholding Murni (Modul 7) | Pipeline Morfologi (Modul 9) |
| :--- | :--- | :--- |
| **Noise Latar Belakang** | Tinggi (watermark & foto ikut bocor) | Sangat Rendah (bersih total) |
| **Keutuhan Karakter** | Banyak goresan patah/terputus | Utuh dan tersambung sempurna |
| **Keseragaman Kontras** | Bergantung pada intensitas lokal | Merata di seluruh bidang dokumen |
| **Keterbacaan Teks** | Rendah - Sedang | Sangat Tinggi (Optimal untuk OCR) |